<div style="background-color: #161b22; padding: 20px; border-radius: 12px; border-left: 6px solid #a371f7; box-shadow: 0 4px 6px rgba(0,0,0,0.3);">
  <h1 style="color: #ffffff; margin: 0; font-family: sans-serif; font-weight: 700; letter-spacing: 1px;">
    🌌 Universal Image Studio <span style="font-size: 0.6em; color: #a371f7; vertical-align: middle; background: #a371f722; padding: 2px 8px; border-radius: 6px;">12 Model Families · 6 LoRA Slots</span>
  </h1>
  <p style="color: #8b949e; margin: 8px 0 0 0; font-family: sans-serif;">
    Z-Image · FLUX.1 · FLUX.2 · Chroma · Qwen-Image · HiDream · Anima · SDXL · Pony · Illustrious · NoobAI · SD 1.5
  </p>
</div>

### 🔴 **Brought to you by [AI With Chucky](https://youtube.com/@AIWithChucky)**
*Subscribe for more AI tutorials, workflows, and optimization tips!*

In [ ]:
#@title 1. Initialize Core Environment
#@markdown This prepares the ephemeral storage, installs ComfyUI, and configures the GGUF integration tools.

import os
import subprocess

LOCAL_WORKSPACE = "/content/ComfyUI"

print("🚀 Initializing Core Architecture...")
if not os.path.exists(LOCAL_WORKSPACE):
    !git clone https://github.com/comfyanonymous/ComfyUI {LOCAL_WORKSPACE} &> /dev/null
    print("   ✓ Core Engine Cloned")
else:
    !cd {LOCAL_WORKSPACE} && git pull &> /dev/null
    print("   ✓ Core Engine Updated")

print("📦 Installing Dependencies (This takes a moment)...")
!cd {LOCAL_WORKSPACE} && pip install xformers!=0.0.18 -r requirements.txt --extra-index-url https://download.pytorch.org/whl/cu121 &> /dev/null

GGUF_NODE_DIR = os.path.join(LOCAL_WORKSPACE, "custom_nodes/ComfyUI-GGUF")
if not os.path.exists(GGUF_NODE_DIR):
    print("🧩 Installing GGUF Processing Nodes...")
    !git clone https://github.com/city96/ComfyUI-GGUF {GGUF_NODE_DIR} &> /dev/null
    !pip install -r {GGUF_NODE_DIR}/requirements.txt &> /dev/null

print("✅ Environment Ready!")

In [ ]:
#@title 2. Model Family & High-Speed Asset Downloader
#@markdown Pick your **model family** — its required text encoder(s) and VAE are fetched automatically. Then paste your model/LoRA links. **Multiple URLs can be separated by commas or new lines.**

import os
import subprocess
import gdown
import urllib.parse

WORKSPACE = "/content/ComfyUI"

MODEL_FAMILY = "Z-Image Turbo" #@param ["Z-Image Turbo", "FLUX.1 (dev/schnell/Krea)", "FLUX.2 dev", "Chroma", "Qwen-Image", "HiDream I1", "Anima", "SDXL", "Pony", "Illustrious", "NoobAI", "SD 1.5"]

# --- Input Resources ---
MODEL_URLS = "" #@param {type:"string"}
LORA_URLS = "" #@param {type:"string"}
#@markdown *Optional: needed for Civitai models that require login (401/403 errors)*
CIVITAI_API_TOKEN = "" #@param {type:"string"}

DIRS = {
    "unet":        os.path.join(WORKSPACE, "models/unet"),
    "checkpoints": os.path.join(WORKSPACE, "models/checkpoints"),
    "clip":        os.path.join(WORKSPACE, "models/clip"),
    "vae":         os.path.join(WORKSPACE, "models/vae"),
    "loras":       os.path.join(WORKSPACE, "models/loras"),
}

# Families whose model file is an all-in-one checkpoint (goes to models/checkpoints)
CHECKPOINT_FAMILIES = {"SDXL", "Pony", "Illustrious", "NoobAI", "SD 1.5"}

HF = "https://huggingface.co"
# Required support files per family: (url, dir_key, save_as) — save_as=None keeps the URL filename
FAMILY_ASSETS = {
    "Z-Image Turbo": [
        (HF + "/Comfy-Org/z_image_turbo/resolve/main/split_files/text_encoders/qwen_3_4b.safetensors", "clip", None),
        (HF + "/Comfy-Org/z_image_turbo/resolve/main/split_files/vae/ae.safetensors", "vae", "z_image_ae.safetensors"),
    ],
    "FLUX.1 (dev/schnell/Krea)": [
        (HF + "/comfyanonymous/flux_text_encoders/resolve/main/clip_l.safetensors", "clip", None),
        (HF + "/comfyanonymous/flux_text_encoders/resolve/main/t5xxl_fp8_e4m3fn_scaled.safetensors", "clip", None),
        (HF + "/lodestones/Chroma/resolve/main/ae.safetensors", "vae", "flux_ae.safetensors"),
    ],
    "FLUX.2 dev": [
        (HF + "/Comfy-Org/flux2-dev/resolve/main/split_files/text_encoders/mistral_3_small_flux2_fp8.safetensors", "clip", None),
        (HF + "/Comfy-Org/flux2-dev/resolve/main/split_files/vae/flux2-vae.safetensors", "vae", None),
    ],
    "Chroma": [
        (HF + "/comfyanonymous/flux_text_encoders/resolve/main/t5xxl_fp8_e4m3fn_scaled.safetensors", "clip", None),
        (HF + "/lodestones/Chroma/resolve/main/ae.safetensors", "vae", "flux_ae.safetensors"),
    ],
    "Qwen-Image": [
        (HF + "/Comfy-Org/Qwen-Image_ComfyUI/resolve/main/split_files/text_encoders/qwen_2.5_vl_7b_fp8_scaled.safetensors", "clip", None),
        (HF + "/Comfy-Org/Qwen-Image_ComfyUI/resolve/main/split_files/vae/qwen_image_vae.safetensors", "vae", None),
    ],
    "HiDream I1": [
        (HF + "/Comfy-Org/HiDream-I1_ComfyUI/resolve/main/split_files/text_encoders/clip_l_hidream.safetensors", "clip", None),
        (HF + "/Comfy-Org/HiDream-I1_ComfyUI/resolve/main/split_files/text_encoders/clip_g_hidream.safetensors", "clip", None),
        (HF + "/Comfy-Org/HiDream-I1_ComfyUI/resolve/main/split_files/text_encoders/t5xxl_fp8_e4m3fn_scaled.safetensors", "clip", None),
        (HF + "/Comfy-Org/HiDream-I1_ComfyUI/resolve/main/split_files/text_encoders/llama_3.1_8b_instruct_fp8_scaled.safetensors", "clip", None),
        (HF + "/lodestones/Chroma/resolve/main/ae.safetensors", "vae", "flux_ae.safetensors"),
    ],
    "Anima": [
        (HF + "/circlestone-labs/Anima/resolve/main/split_files/text_encoders/qwen_3_06b_base.safetensors", "clip", None),
        (HF + "/circlestone-labs/Anima/resolve/main/split_files/vae/qwen_image_vae.safetensors", "vae", None),
    ],
    # Checkpoint families: everything (CLIP + VAE) is inside the checkpoint file itself
    "SDXL": [], "Pony": [], "Illustrious": [], "NoobAI": [], "SD 1.5": [],
}

print("⚡ Configuring Aria2c Accelerator...")
subprocess.run(['apt-get', '-y', 'install', '-qq', 'aria2'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

def download_file(url, target_dir, save_as=None):
    try:
        os.makedirs(target_dir, exist_ok=True)
        before_files = set(os.listdir(target_dir))

        if save_as and os.path.exists(os.path.join(target_dir, save_as)):
            print(f"   ⏭️ Already in library — skipped: \033[96m{save_as}\033[0m")
            return

        if "drive.google.com" in url:
            print(f"   📥 Downloading from Drive...")
            gdown.download(url, output=target_dir + '/', quiet=False, fuzzy=True)
        else:
            print(f"   📥 Fetching: {(save_as or url.split('/')[-1])[:50]}...")
            parsed_url = urllib.parse.urlparse(url)
            filename = os.path.basename(parsed_url.path)

            aria2_cmd = [
                "aria2c", "--console-log-level=error", "--summary-interval=10",
                "-c", "-x", "16", "-s", "16", "-k", "1M"
            ]

            is_civitai = "civitai" in parsed_url.netloc.lower()
            has_extension = os.path.splitext(filename)[1].lower() in (
                ".safetensors", ".gguf", ".ckpt", ".pt", ".pth", ".bin", ".sft", ".zip")

            if is_civitai and "/api/download/" in parsed_url.path and CIVITAI_API_TOKEN.strip():
                sep = "&" if parsed_url.query else "?"
                url = f"{url}{sep}token={CIVITAI_API_TOKEN.strip()}"

            if save_as:
                aria2_cmd.extend(["-o", save_as, url, "-d", target_dir])
            elif is_civitai or not has_extension:
                # Let the server name the file (follows redirects, reads Content-Disposition)
                aria2_cmd.extend(["--content-disposition", url, "-d", target_dir])
            else:
                aria2_cmd.extend(["-o", filename, url, "-d", target_dir])

            subprocess.run(aria2_cmd, check=True)

        after_files = set(os.listdir(target_dir))
        new_files = after_files - before_files

        if new_files:
            downloaded_file = list(new_files)[0]
            print(f"   ✅ Saved as: \033[96m{downloaded_file}\033[0m")
        else:
            print(f"   ⏭️ Already in library — skipped (existing file kept)")

    except Exception as e:
        print(f"   ❌ Failed: {url}\n      Error: {e}\n")

def process_downloads(urls_str, target_dir):
    if not urls_str.strip(): return
    url_list = [u.strip() for u in urls_str.replace(',', '\n').split('\n') if u.strip()]
    os.makedirs(target_dir, exist_ok=True)
    print(f"\n📂 Directory: {os.path.basename(target_dir)}")
    for url in url_list:
        download_file(url, target_dir)

# --- Route the main model file based on family type ---
model_dir_key = "checkpoints" if MODEL_FAMILY in CHECKPOINT_FAMILIES else "unet"
print(f"\n🧬 Family: \033[95m{MODEL_FAMILY}\033[0m → model files go to \033[96mmodels/{model_dir_key}\033[0m")

process_downloads(MODEL_URLS, DIRS[model_dir_key])
process_downloads(LORA_URLS,  DIRS["loras"])

# --- Auto-fetch this family's required support files ---
assets = FAMILY_ASSETS[MODEL_FAMILY]
if assets:
    print(f"\n🔧 Fetching required support files for {MODEL_FAMILY}...")
    for url, dir_key, save_as in assets:
        download_file(url, DIRS[dir_key], save_as=save_as)
else:
    print(f"\n🔧 {MODEL_FAMILY} checkpoints are all-in-one — no extra support files needed.")

print("\n✅ All assets secured!")

# --- Asset Library Inventory ---
def _human_size(num_bytes):
    for unit in ["B", "KB", "MB", "GB"]:
        if num_bytes < 1024 or unit == "GB":
            return f"{num_bytes:.1f} {unit}" if unit != "B" else f"{num_bytes} B"
        num_bytes /= 1024

import html as _htmlmod
import json as _jsonmod
try:
    from IPython.display import display as _display, HTML as _HTML
    _HAVE_IPY = True
except Exception:
    _HAVE_IPY = False

_inv = ['<div style="background:#161b22;border:1px solid #30363d;border-radius:10px;'
        'padding:14px 18px;font-family:monospace;margin:8px 0;max-width:640px;">'
        '<div style="color:#2ecc71;font-weight:bold;font-size:14px;">📋 ASSET LIBRARY</div>'
        '<div style="color:#8b949e;font-size:11px;margin-bottom:8px;">'
        'Click any name to copy it, then paste into the Generation cell</div>']
for label, key in [("🧠 Diffusion Models (→ MODEL_FILENAME)", "unet"),
                   ("📦 Checkpoints (→ MODEL_FILENAME)", "checkpoints"),
                   ("🎨 LoRAs (→ LORA_1..6_FILENAME)", "loras")]:
    folder = DIRS[key]
    entries = sorted(f for f in os.listdir(folder)) if os.path.exists(folder) else []
    entries = [f for f in entries if not f.endswith(".aria2")]
    if not entries and key == ("unet" if MODEL_FAMILY in CHECKPOINT_FAMILIES else "checkpoints"):
        continue  # hide the irrelevant empty section for the current family
    _inv.append(f'<div style="color:#e6edf3;font-size:12px;margin:10px 0 4px 0;">{label}:</div>')
    if not entries:
        _inv.append('<div style="color:#8b949e;font-size:12px;margin-left:12px;">(empty)</div>')
    for f in entries:
        size = _human_size(os.path.getsize(os.path.join(folder, f)))
        safe_html = _htmlmod.escape(f)
        safe_js = _htmlmod.escape(_jsonmod.dumps(f), quote=True)
        _inv.append(
            f'<div style="margin:3px 0 3px 12px;">'
            f'<code onclick="navigator.clipboard.writeText({safe_js});'
            f'var b=this.nextElementSibling;b.textContent=\'✓ copied!\';'
            f'setTimeout(function(){{b.textContent=\'({size})\';}},1500);" '
            f'style="color:#58d6ff;background:#0d1117;border:1px solid #30363d;'
            f'border-radius:6px;padding:3px 8px;cursor:pointer;font-size:13px;" '
            f'title="Click to copy">{safe_html}</code> '
            f'<span style="color:#8b949e;font-size:11px;">({size})</span></div>')
_inv.append('</div>')
if _HAVE_IPY:
    _display(_HTML("".join(_inv)))


In [ ]:
#@title 3. Image Generation
#@markdown Pick the **same model family** you downloaded for in Cell 2, enter your prompt, and go. With **AUTO_DEFAULTS** on, the recommended steps/CFG/sampler/shift for the family are applied automatically.

# --- Model Selection ---
MODEL_FAMILY = "Z-Image Turbo" #@param ["Z-Image Turbo", "FLUX.1 (dev/schnell/Krea)", "FLUX.2 dev", "Chroma", "Qwen-Image", "HiDream I1", "Anima", "SDXL", "Pony", "Illustrious", "NoobAI", "SD 1.5"]
MODEL_FILENAME = "" #@param {type:"string"}

#@markdown ---
#@markdown ### 🎨 LoRA Stack — a slot is skipped if its filename is empty/"none" **or its strength is 0** (easy on/off without retyping names)
LORA_1_FILENAME = "" #@param {type:"string"}
LORA_1_STRENGTH = 1.0 #@param {type:"slider", min:0.0, max:2.0, step:0.05}
LORA_2_FILENAME = "" #@param {type:"string"}
LORA_2_STRENGTH = 1.0 #@param {type:"slider", min:0.0, max:2.0, step:0.05}
LORA_3_FILENAME = "" #@param {type:"string"}
LORA_3_STRENGTH = 1.0 #@param {type:"slider", min:0.0, max:2.0, step:0.05}
LORA_4_FILENAME = "" #@param {type:"string"}
LORA_4_STRENGTH = 1.0 #@param {type:"slider", min:0.0, max:2.0, step:0.05}
LORA_5_FILENAME = "" #@param {type:"string"}
LORA_5_STRENGTH = 1.0 #@param {type:"slider", min:0.0, max:2.0, step:0.05}
LORA_6_FILENAME = "" #@param {type:"string"}
LORA_6_STRENGTH = 1.0 #@param {type:"slider", min:0.0, max:2.0, step:0.05}
#@markdown ---

# --- Generation Settings ---
PROMPT = "" #@param {type:"string"}
NEGATIVE_PROMPT = "blurry, low quality, deformed, artifacts" #@param {type:"string"}

#@markdown ---
#@markdown ### 📐 Resolution — every option shows its exact size, pick and done
RESOLUTION = "16:9 · FHD · 1920x1080" #@param ["Custom (use sliders below)", "1:1 · SD · 720x720", "1:1 · HD · 896x896", "1:1 · FHD · 1080x1080", "16:9 · SD · 1280x720", "16:9 · HD · 1600x896", "16:9 · FHD · 1920x1080", "9:16 · SD · 720x1280", "9:16 · HD · 896x1600", "9:16 · FHD · 1080x1920", "4:3 · SD · 960x720", "4:3 · HD · 1184x888", "4:3 · FHD · 1440x1080", "3:4 · SD · 720x960", "3:4 · HD · 888x1184", "3:4 · FHD · 1080x1440", "3:2 · SD · 1080x720", "3:2 · HD · 1344x896", "3:2 · FHD · 1632x1088", "2:3 · SD · 720x1080", "2:3 · HD · 896x1344", "2:3 · FHD · 1088x1632", "21:9 · SD · 1344x576", "21:9 · HD · 1680x720", "21:9 · FHD · 2016x864"]
#@markdown *Custom size (only used when RESOLUTION is "Custom"): use the sliders, or type exact numbers in the override boxes (0 = use slider). Tip: SD 1.5 works best around 512–768.*
WIDTH = 1024 #@param {type:"slider", min:512, max:2048, step:8}
HEIGHT = 1024 #@param {type:"slider", min:512, max:2048, step:8}
WIDTH_OVERRIDE = 0 #@param {type:"integer"}
HEIGHT_OVERRIDE = 0 #@param {type:"integer"}
#@markdown ---
BATCH_SIZE = 1 #@param {type:"slider", min:1, max:4, step:1}

# --- Sampler Settings ---
#@markdown ### ⚙️ Sampling — leave AUTO_DEFAULTS on to use each family's recommended settings; turn it off to use the manual values below
AUTO_DEFAULTS = True #@param {type:"boolean"}
STEPS = 9 #@param {type:"slider", min:1, max:60, step:1}
CFG = 1.0 #@param {type:"number"}
SAMPLER_NAME = "res_multistep" #@param ["euler", "euler_cfg_pp", "euler_ancestral", "euler_ancestral_cfg_pp", "heun", "heunpp2", "dpm_2", "dpm_2_ancestral", "lms", "dpm_fast", "dpm_adaptive", "dpmpp_2s_ancestral", "dpmpp_2s_ancestral_cfg_pp", "dpmpp_sde", "dpmpp_sde_gpu", "dpmpp_2m", "dpmpp_2m_cfg_pp", "dpmpp_2m_sde", "dpmpp_2m_sde_gpu", "dpmpp_3m_sde", "dpmpp_3m_sde_gpu", "ddpm", "lcm", "ipndm", "ipndm_v", "deis", "res_multistep", "res_multistep_cfg_pp", "res_multistep_ancestral", "res_multistep_ancestral_cfg_pp", "gradient_estimation", "gradient_estimation_cfg_pp", "er_sde", "seeds_2", "seeds_3", "sa_solver", "sa_solver_pece", "ddim", "uni_pc", "uni_pc_bh2"] {allow-input: true}
SCHEDULER = "beta" #@param ["normal", "karras", "exponential", "sgm_uniform", "simple", "ddim_uniform", "beta", "linear_quadratic", "kl_optimal"] {allow-input: true}
SHIFT = 3.0 #@param {type:"slider", min:1.0, max:10.0, step:0.5}
FLUX_GUIDANCE = 3.5 #@param {type:"number"}
SEED = 0 #@param {type:"integer"}

import sys
import os
import json
import time
import random
import subprocess
import urllib.request

try:
    from IPython.display import display as _display, HTML as _HTML, Image as IPImage
    _HAVE_IPY = True
except Exception:
    _HAVE_IPY = False

WORKSPACE = "/content/ComfyUI"
if os.path.isdir(WORKSPACE):
    os.chdir(WORKSPACE)

if SEED == 0:
    SEED = random.randint(1, 1125899906842624)

# ============================================================
# MODEL FAMILY REGISTRY
# loader: "unet" (separate model + encoders + VAE) or "checkpoint" (all-in-one)
# clip:   encoder loader spec | None for checkpoints
# vae:    VAE filename | None for checkpoints
# latent: empty-latent node class
# shift:  (node_class, recommended_shift) | None
# guidance: recommended FluxGuidance value | None
# sampling: "ksampler" | "flux2" (custom sampling path)
# defaults: recommended steps/cfg/sampler/scheduler
# ============================================================
FAMILIES = {
    "Z-Image Turbo": dict(
        loader="unet", clip=("CLIPLoader", "qwen_3_4b.safetensors", "lumina2"),
        vae="z_image_ae.safetensors", latent="EmptySD3LatentImage",
        shift=("ModelSamplingAuraFlow", 3.0), guidance=None, sampling="ksampler",
        defaults=dict(steps=9, cfg=1.0, sampler="res_multistep", scheduler="beta"),
        tip="Distilled turbo: keep CFG at 1.0. Sweet spot 9-14 steps. Shift 2.0-2.5 favors fine detail."),
    "FLUX.1 (dev/schnell/Krea)": dict(
        loader="unet", clip=("DualCLIPLoader", "clip_l.safetensors", "t5xxl_fp8_e4m3fn_scaled.safetensors", "flux"),
        vae="flux_ae.safetensors", latent="EmptySD3LatentImage",
        shift=None, guidance=3.5, sampling="ksampler",
        defaults=dict(steps=20, cfg=1.0, sampler="euler", scheduler="simple"),
        tip="dev/Krea-dev: 20-28 steps, guidance 3.5. schnell: only 4 steps! CFG always stays 1.0."),
    "FLUX.2 dev": dict(
        loader="unet", clip=("CLIPLoader", "mistral_3_small_flux2_fp8.safetensors", "flux2"),
        vae="flux2-vae.safetensors", latent="EmptyFlux2LatentImage",
        shift=None, guidance=4.0, sampling="flux2",
        defaults=dict(steps=20, cfg=1.0, sampler="euler", scheduler="simple"),
        tip="32B model — use a GGUF quant (Q4 or smaller) on Colab. Negative prompt is not used by this family."),
    "Chroma": dict(
        loader="unet", clip=("CLIPLoader", "t5xxl_fp8_e4m3fn_scaled.safetensors", "chroma"),
        vae="flux_ae.safetensors", latent="EmptySD3LatentImage",
        shift=("ModelSamplingAuraFlow", 1.0), guidance=None, sampling="ksampler",
        defaults=dict(steps=26, cfg=3.5, sampler="euler", scheduler="beta"),
        tip="Real CFG model: negative prompt works! Descriptive natural-language negatives work best."),
    "Qwen-Image": dict(
        loader="unet", clip=("CLIPLoader", "qwen_2.5_vl_7b_fp8_scaled.safetensors", "qwen_image"),
        vae="qwen_image_vae.safetensors", latent="EmptySD3LatentImage",
        shift=("ModelSamplingAuraFlow", 3.1), guidance=None, sampling="ksampler",
        defaults=dict(steps=20, cfg=2.5, sampler="euler", scheduler="simple"),
        tip="Excellent at text-in-image (incl. Chinese). With a lightning/distill LoRA: 8 steps, CFG 1.0."),
    "HiDream I1": dict(
        loader="unet", clip=("QuadrupleCLIPLoader", "clip_l_hidream.safetensors", "clip_g_hidream.safetensors", "t5xxl_fp8_e4m3fn_scaled.safetensors", "llama_3.1_8b_instruct_fp8_scaled.safetensors"),
        vae="flux_ae.safetensors", latent="EmptySD3LatentImage",
        shift=("ModelSamplingSD3", 6.0), guidance=None, sampling="ksampler",
        defaults=dict(steps=28, cfg=1.0, sampler="lcm", scheduler="normal"),
        tip="17B + 4 text encoders = VRAM heavy; use fp8/GGUF. dev: 28 steps CFG 1 | full: 50 steps CFG 5 | fast: 16 steps CFG 1."),
    "Anima": dict(
        loader="unet", clip=("CLIPLoader", "qwen_3_06b_base.safetensors", "stable_diffusion"),
        vae="qwen_image_vae.safetensors", latent="EmptyLatentImage",
        shift=None, guidance=None, sampling="ksampler",
        defaults=dict(steps=30, cfg=4.0, sampler="euler", scheduler="simple"),
        tip="Anime/illustration only (no realism). Danbooru tags AND natural language both work. With anima-turbo LoRA: 8 steps, CFG 1."),
    "SDXL": dict(
        loader="checkpoint", clip=None, vae=None, latent="EmptyLatentImage",
        shift=None, guidance=None, sampling="ksampler",
        defaults=dict(steps=28, cfg=6.0, sampler="dpmpp_2m", scheduler="karras"),
        tip="Native ~1024px. Real CFG model: negative prompt matters. LoRAs apply to model + text encoder."),
    "Pony": dict(
        loader="checkpoint", clip=None, vae=None, latent="EmptyLatentImage",
        shift=None, guidance=None, sampling="ksampler",
        defaults=dict(steps=25, cfg=7.0, sampler="euler_ancestral", scheduler="normal"),
        tip="Start prompts with: score_9, score_8_up, score_7_up. Source tags like source_anime help."),
    "Illustrious": dict(
        loader="checkpoint", clip=None, vae=None, latent="EmptyLatentImage",
        shift=None, guidance=None, sampling="ksampler",
        defaults=dict(steps=28, cfg=5.5, sampler="euler_ancestral", scheduler="normal"),
        tip="Danbooru-tag prompting (1girl, solo, ...). masterpiece/best quality tags help."),
    "NoobAI": dict(
        loader="checkpoint", clip=None, vae=None, latent="EmptyLatentImage",
        shift=None, guidance=None, sampling="ksampler",
        defaults=dict(steps=28, cfg=5.5, sampler="euler_ancestral", scheduler="normal"),
        tip="Illustrious-based, deep Danbooru/e621 tag knowledge. Artist tags are very strong."),
    "SD 1.5": dict(
        loader="checkpoint", clip=None, vae=None, latent="EmptyLatentImage",
        shift=None, guidance=None, sampling="ksampler",
        defaults=dict(steps=25, cfg=7.0, sampler="dpmpp_2m", scheduler="karras"),
        tip="Native 512px — use SD-tier resolutions or ~512-768 custom; going higher causes doubling artifacts."),
}

FAM = FAMILIES[MODEL_FAMILY]

# --- Resolve resolution ---
if "Custom" not in RESOLUTION:
    ratio_key, _quality, _dims = [p.strip() for p in RESOLUTION.split("·")]
    WIDTH, HEIGHT = (int(v) for v in _dims.split("x"))
    _res_source = f"Preset: {ratio_key} @ {_quality} — sliders are IGNORED"
else:
    ratio_key = "custom"
    WIDTH = int(WIDTH_OVERRIDE) if int(WIDTH_OVERRIDE) > 0 else int(WIDTH)
    HEIGHT = int(HEIGHT_OVERRIDE) if int(HEIGHT_OVERRIDE) > 0 else int(HEIGHT)
    WIDTH, HEIGHT = max(64, WIDTH), max(64, HEIGHT)
    if WIDTH % 8 or HEIGHT % 8:
        WIDTH, HEIGHT = (WIDTH // 8) * 8, (HEIGHT // 8) * 8
        _res_source = "Custom sliders (snapped to nearest multiple of 8)"
    else:
        _res_source = "Custom sliders"

# --- Resolve sampling settings (family defaults vs manual) ---
if AUTO_DEFAULTS:
    _d = FAM["defaults"]
    steps_val, cfg_val = _d["steps"], _d["cfg"]
    sampler_val, scheduler_val = _d["sampler"], _d["scheduler"]
    shift_val = FAM["shift"][1] if FAM["shift"] else None
    guidance_val = FAM["guidance"]
    _settings_source = "AUTO (family recommended)"
else:
    steps_val, cfg_val = STEPS, CFG
    sampler_val, scheduler_val = SAMPLER_NAME, SCHEDULER
    shift_val = SHIFT if FAM["shift"] else None
    guidance_val = FLUX_GUIDANCE if FAM["guidance"] is not None else None
    _settings_source = "MANUAL"

# --- Validate inputs early ---
if not MODEL_FILENAME.strip():
    raise ValueError("MODEL_FILENAME is empty — copy a name from the Asset Library in Cell 2.")
if FAM["loader"] == "checkpoint" and MODEL_FILENAME.lower().endswith(".gguf"):
    raise ValueError(f"{MODEL_FAMILY} uses all-in-one checkpoints — GGUF is not supported for this family. Use a .safetensors checkpoint.")

# --- LoRA stack ---
LORA_SLOTS = [
    (LORA_1_FILENAME, LORA_1_STRENGTH), (LORA_2_FILENAME, LORA_2_STRENGTH),
    (LORA_3_FILENAME, LORA_3_STRENGTH), (LORA_4_FILENAME, LORA_4_STRENGTH),
    (LORA_5_FILENAME, LORA_5_STRENGTH), (LORA_6_FILENAME, LORA_6_STRENGTH),
]
active_loras = [(name.strip(), strength) for name, strength in LORA_SLOTS
                if name.strip() and name.strip().lower() != "none" and strength != 0]
disabled_loras = [name.strip() for name, strength in LORA_SLOTS
                  if name.strip() and name.strip().lower() != "none" and strength == 0]

# --- Pre-flight file checks: fail HERE with a clear message, not with a cryptic 400 from the server ---
def _check_file_exists(kind, filename, folder):
    if not os.path.isdir(folder):
        return  # downloader hasn't run in this session; let the server validate
    available = sorted(f for f in os.listdir(folder) if not f.endswith(".aria2"))
    if filename not in available:
        listing = "\n".join(f"   • {f}" for f in available) if available else "   (folder is empty)"
        hint = ""
        if len(filename) > 80 or "," in filename:
            hint = "\n   💡 This looks like a prompt pasted into the wrong field — check your form inputs!"
        raise ValueError(
            f"{kind} '{filename[:100]}' not found in {os.path.basename(folder)}/.{hint}\n"
            f"   Available files:\n{listing}\n"
            f"   → Copy the exact name from the Asset Library in Cell 2.")

_model_folder = os.path.join(WORKSPACE, "models/checkpoints" if FAM["loader"] == "checkpoint" else "models/unet")
_check_file_exists("Model", MODEL_FILENAME.strip(), _model_folder)
for _lname, _ in active_loras:
    _check_file_exists("LoRA", _lname, os.path.join(WORKSPACE, "models/loras"))

# ============================================================
# BUILD THE WORKFLOW GRAPH
# ============================================================
prompt_workflow = {}

# 1) Base model loader
if FAM["loader"] == "checkpoint":
    prompt_workflow["10"] = {"inputs": {"ckpt_name": MODEL_FILENAME}, "class_type": "CheckpointLoaderSimple"}
    model_link, clip_link, vae_link = ["10", 0], ["10", 1], ["10", 2]
else:
    if MODEL_FILENAME.lower().endswith(".gguf"):
        prompt_workflow["10"] = {"inputs": {"unet_name": MODEL_FILENAME}, "class_type": "UnetLoaderGGUF"}
    else:
        prompt_workflow["10"] = {"inputs": {"unet_name": MODEL_FILENAME, "weight_dtype": "default"}, "class_type": "UNETLoader"}
    model_link = ["10", 0]

    c = FAM["clip"]
    if c[0] == "CLIPLoader":
        prompt_workflow["11"] = {"inputs": {"clip_name": c[1], "type": c[2], "device": "default"}, "class_type": "CLIPLoader"}
    elif c[0] == "DualCLIPLoader":
        prompt_workflow["11"] = {"inputs": {"clip_name1": c[1], "clip_name2": c[2], "type": c[3], "device": "default"}, "class_type": "DualCLIPLoader"}
    else:  # QuadrupleCLIPLoader
        prompt_workflow["11"] = {"inputs": {"clip_name1": c[1], "clip_name2": c[2], "clip_name3": c[3], "clip_name4": c[4]}, "class_type": "QuadrupleCLIPLoader"}
    clip_link = ["11", 0]

    prompt_workflow["12"] = {"inputs": {"vae_name": FAM["vae"]}, "class_type": "VAELoader"}
    vae_link = ["12", 0]

# 2) LoRA chain (checkpoint families also patch the text encoder — full LoraLoader)
for idx, (lora_name, lora_strength) in enumerate(active_loras):
    node_id = str(20 + idx)
    if FAM["loader"] == "checkpoint":
        prompt_workflow[node_id] = {
            "inputs": {"lora_name": lora_name, "strength_model": lora_strength, "strength_clip": lora_strength,
                       "model": model_link, "clip": clip_link},
            "class_type": "LoraLoader"}
        model_link, clip_link = [node_id, 0], [node_id, 1]
    else:
        prompt_workflow[node_id] = {
            "inputs": {"lora_name": lora_name, "strength_model": lora_strength, "model": model_link},
            "class_type": "LoraLoaderModelOnly"}
        model_link = [node_id, 0]

# 3) Sampling-shift wrapper (family specific)
if FAM["shift"]:
    prompt_workflow["30"] = {"inputs": {"shift": shift_val, "model": model_link}, "class_type": FAM["shift"][0]}
    model_link = ["30", 0]

# 4) Prompt encoding (+ FluxGuidance where the family uses it)
prompt_workflow["31"] = {"inputs": {"text": PROMPT, "clip": clip_link}, "class_type": "CLIPTextEncode"}
positive_link = ["31", 0]
prompt_workflow["32"] = {"inputs": {"text": NEGATIVE_PROMPT, "clip": clip_link}, "class_type": "CLIPTextEncode"}
negative_link = ["32", 0]
if guidance_val is not None:
    prompt_workflow["33"] = {"inputs": {"guidance": guidance_val, "conditioning": positive_link}, "class_type": "FluxGuidance"}
    positive_link = ["33", 0]

# 5) Empty latent (family-specific latent space)
prompt_workflow["40"] = {"inputs": {"width": WIDTH, "height": HEIGHT, "batch_size": BATCH_SIZE}, "class_type": FAM["latent"]}

# 6) Sampling
if FAM["sampling"] == "flux2":
    prompt_workflow["50"] = {"inputs": {"noise_seed": SEED}, "class_type": "RandomNoise"}
    prompt_workflow["51"] = {"inputs": {"model": model_link, "conditioning": positive_link}, "class_type": "BasicGuider"}
    prompt_workflow["52"] = {"inputs": {"sampler_name": sampler_val}, "class_type": "KSamplerSelect"}
    prompt_workflow["53"] = {"inputs": {"steps": steps_val, "width": WIDTH, "height": HEIGHT}, "class_type": "Flux2Scheduler"}
    prompt_workflow["54"] = {"inputs": {"noise": ["50", 0], "guider": ["51", 0], "sampler": ["52", 0],
                                        "sigmas": ["53", 0], "latent_image": ["40", 0]},
                             "class_type": "SamplerCustomAdvanced"}
    latent_out = ["54", 0]
else:
    prompt_workflow["54"] = {"inputs": {"seed": SEED, "steps": steps_val, "cfg": cfg_val,
                                        "sampler_name": sampler_val, "scheduler": scheduler_val, "denoise": 1,
                                        "model": model_link, "positive": positive_link, "negative": negative_link,
                                        "latent_image": ["40", 0]},
                             "class_type": "KSampler"}
    latent_out = ["54", 0]

# 7) Decode & save
prompt_workflow["60"] = {"inputs": {"samples": latent_out, "vae": vae_link}, "class_type": "VAEDecode"}
prompt_workflow["61"] = {"inputs": {"filename_prefix": "universal", "images": ["60", 0]}, "class_type": "SaveImage"}

# ============================================================
# STATUS DISPLAY
# ============================================================
if _HAVE_IPY:
    _max_box = 220
    _scale = _max_box / max(WIDTH, HEIGHT)
    _bw, _bh = int(WIDTH * _scale), int(HEIGHT * _scale)
    _card = ('<div style="background:#161b22;border:1px solid #2ecc71;border-radius:10px;padding:14px 18px;'
             'font-family:monospace;display:inline-block;margin:6px 0;">'
             '<div style="color:#2ecc71;font-size:15px;font-weight:bold;">🧬 ' + MODEL_FAMILY
             + ' &nbsp;|&nbsp; 📐 ' + str(WIDTH) + ' × ' + str(HEIGHT) + ' px</div>'
             '<div style="color:#8b949e;font-size:12px;margin:2px 0 10px 0;">' + _res_source + '</div>'
             '<div style="width:' + str(_bw) + 'px;height:' + str(_bh) + 'px;'
             'background:linear-gradient(135deg,#1f6feb33,#2ecc7133);border:2px dashed #2ecc71;'
             'border-radius:4px;display:flex;align-items:center;justify-content:center;'
             'color:#e6edf3;font-size:12px;">'
             + (ratio_key if ratio_key != "custom" else str(WIDTH) + "x" + str(HEIGHT)) + '</div></div>')
    _display(_HTML(_card))

print(f"\033[95m💡 {MODEL_FAMILY}: {FAM['tip']}\033[0m")
_line = f"➜ Settings [{_settings_source}] | Steps: {steps_val} | CFG: {cfg_val} | {sampler_val}/{scheduler_val}"
if shift_val is not None: _line += f" | Shift: {shift_val}"
if guidance_val is not None: _line += f" | Guidance: {guidance_val}"
print(f"\033[94m{_line}\033[0m")
print(f"\033[94m➜ Size: {WIDTH}x{HEIGHT} ({ratio_key}) | Batch: {BATCH_SIZE} | Seed: {SEED}\033[0m")

if active_loras:
    print("\033[95m➜ Active LoRA Stack:\033[0m")
    for i, (lora_name, lora_strength) in enumerate(active_loras, 1):
        print(f"   {i}. {lora_name}  (strength: {lora_strength})")
else:
    print("\033[93m➜ No LoRAs active — running base model only.\033[0m")
if disabled_loras:
    for lora_name in disabled_loras:
        print(f"\033[90m   ⏸ Disabled (strength 0): {lora_name}\033[0m")
if MODEL_FAMILY == "FLUX.2 dev" and NEGATIVE_PROMPT.strip():
    print("\033[90m   ℹ FLUX.2 does not use a negative prompt — it will be ignored.\033[0m")

# === SUBMIT TO API ===
print("\n🔌 Checking ComfyUI Server Status...")
def start_server():
    req = urllib.request.Request("http://127.0.0.1:8188")
    try:
        urllib.request.urlopen(req)
        print("   🟢 Server is already running.")
    except:
        print("   🚀 Starting ComfyUI Server in background...")
        subprocess.Popen([sys.executable, "main.py"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        while True:
            try:
                urllib.request.urlopen(req)
                print("   🟢 Server is now up and ready!")
                break
            except:
                time.sleep(2)

start_server()

p = {"prompt": prompt_workflow}
data = json.dumps(p).encode('utf-8')
req = urllib.request.Request("http://127.0.0.1:8188/prompt", data=data)

print("   📥 Submitting workflow to API...")
try:
    response = urllib.request.urlopen(req)
    prompt_id = json.loads(response.read())['prompt_id']
except urllib.error.HTTPError as e:
    body = e.read().decode('utf-8', errors='replace')[:1500]
    print(f"   ❌ API Error {e.code}: {body}")
    raise
except Exception as e:
    print(f"   ❌ API Error: {e}")
    raise

print("   ✨ Processing and Sampling (Check ComfyUI server logs if stuck)...")
while True:
    try:
        history_req = urllib.request.Request(f"http://127.0.0.1:8188/history/{prompt_id}")
        history_res = urllib.request.urlopen(history_req)
        history_data = json.loads(history_res.read())
        if prompt_id in history_data:
            outputs = history_data[prompt_id]['outputs']
            break
    except:
        pass
    time.sleep(1)

print("   🖼️ Decoding Final Masterpiece...")
for node_id, node_output in outputs.items():
    if 'images' in node_output:
        for image in node_output['images']:
            filename = image['filename']
            img_path = os.path.join(WORKSPACE, "output", filename)
            print(f"\033[92m✓ Saved: {filename}\033[0m")
            if _HAVE_IPY:
                _display(IPImage(filename=img_path))


In [ ]:
 #@title 4. Export & Download Results
#@markdown Run this to instantly zip and download all the generated images from this session.

import os
from google.colab import files

OUTPUT_DIR = "/content/ComfyUI/output"
ZIP_NAME = "/content/Z_Image_Artworks.zip"

if os.path.exists(OUTPUT_DIR) and len(os.listdir(OUTPUT_DIR)) > 0:
    print("🗜️ Zipping generated artworks...")
    !zip -j -q {ZIP_NAME} {OUTPUT_DIR}/*.png
    print("📥 Initiating download...")
    files.download(ZIP_NAME)
else:
    print("⚠️ No images found in the output directory yet!")